# Config 1: Gemma, CPU-only, zero-shot

This is the floor of the three-config comparison: an off-the-shelf small
open-weight model, no fine-tuning, no tool access, running entirely on
ordinary CPU hardware -- the air-gapped baseline. `config2-qlora-gpu/`
fine-tunes the same base model; `config3-frontier-skills/` gives a
frontier model SAS-metadata tool access. See the repo root's `PLAN.md`
for the full three-way framing.

This notebook is a runnable copy of `SETUP.md` in this folder (section
numbers match). Unlike config2's notebook, this one is meant to run
right here on this box, not in Colab -- config1's whole point is that it
needs nothing but ordinary CPU hardware, and the model server is already
set up locally.

Run top to bottom: confirm the model server -> install deps -> run
documentation generation (single program or the full eval batch) ->
*(optional)* push results to SAS via ODA.

## 1. The model runtime (already set up on this box)

The actual model weights + Ollama server live OUTSIDE this folder, at
`/internal/e2b-gemma/` -- that's infrastructure (a 9.6 GB directory of
model blobs and platform binaries), not project code, so it isn't
duplicated here. See `/internal/e2b-gemma/README.md` for the full
writeup.

`gemma3:1b` (815 MB, text-only) is what config1 uses by default. It ran
end-to-end on this box's CPU-only AMD A8-6410 APU, 4.7 GB RAM: ~11 tok/s
prompt processing, ~4.5 tok/s generation, roughly 3-6 minutes per program
in JSON mode. Budget accordingly -- this is a batch/offline tool, not an
interactive one, on hardware like this.

**On a different machine**, install Ollama yourself (no sudo required --
see Ollama's docs for the tarball install) and `ollama pull gemma3:1b`
instead of running the cell below.

In [ ]:
# Confirm the server is up before running anything below.
!curl -s http://127.0.0.1:11434/api/version

In [ ]:
# Only needed if the check above didn't return a version (e.g. after a
# reboot) -- the server is already running on this box otherwise. Starts
# it in the background and detaches; safe to leave commented out.
# !nohup /internal/e2b-gemma/serve.sh > /internal/e2b-gemma/server.log 2>&1 &
# !disown

## 2. Python dependencies

Just `requests` for the Ollama HTTP API. The ODA push step (section 4
below) needs `saspy` + `pandas` too -- those are already installed into
`/internal/venvs/main` on this box, which is why that step below invokes
that venv's `python3` directly instead of the kernel running this
notebook.

In [ ]:
!pip install -r requirements.txt

## 3. Run it

Zero-shot: no fine-tuning happens here (that's config2, which needs a
GPU). Ollama's `format: "json"` constrains decoding to valid JSON
syntax, which helps a lot on a 1B model -- `schema.py`'s `validate()`
still checks the CONTENT shape on top of that.

Each program run writes `<name>.pred.json` (the structured dictionary),
`<name>.meta.json` (timing, in the shape `results/run_eval.py` expects
for its "Time per program"/"Cost per program" columns), `<name>.raw.txt`
(the model's raw response, kept even on a parse failure, for debugging),
and `<name>.guardrail.json` (hallucination check against `extract.py`'s
static source scan). Every run also upserts into a local, offline JSONL
catalog (`catalog.py`) that `push_to_oda.py` can later push to SAS.

Single program first, to see the shape of the output:

In [ ]:
!python3 document_sas.py ../eval-programs/programs/prog900_estab.sas --out ../results/preds/config1-gemma-cpu --catalog ../results/catalog/config1-gemma-cpu

Then the whole eval set -- one program at a time, in sequence (this box
has no GPU and no request batching), so budget roughly 1-2 hours for the
20 held-out `eval-programs/`. A per-file failure (bad file, Ollama
hiccup, timeout) is logged and skipped rather than aborting the run. The
`--out`/`--catalog` paths below match `../results/preds/<config>` --
the convention `results/run_eval.py` expects and config2's notebook also
writes to, so both configs' outputs are directly comparable:

In [ ]:
!python3 document_sas.py --dir ../eval-programs/programs --out ../results/preds/config1-gemma-cpu --catalog ../results/catalog/config1-gemma-cpu

## 4. (Optional) SAS OnDemand for Academics (ODA) credentials

Only needed if you want to push the generated dictionary into SAS as
real datasets (`push_to_oda.py`), or to harvest ground-truth
`dictionary.columns` metadata for a check config1 deliberately doesn't
use (config1 has no ground-truth access on purpose -- that's config3's
job). Skip this section entirely if you only want `document_sas.py`'s
local JSON/CSV output.

**Get a free ODA account** first if you don't have one (SAS OnDemand for
Academics signup -- no cost, academic/non-commercial use).

**Create `~/.authinfo` yourself, in your own terminal** -- not in a
notebook cell that gets saved:
```bash
echo "oda user YOUR_ODA_EMAIL password YOUR_ODA_PASSWORD" >> ~/.authinfo
chmod 600 ~/.authinfo
```
Then check your ODA region matches `config/sascfg_personal.py`'s
`iomhost` list (already filled in for US-region/usw2 -- see `SETUP.md`
for the other two regions' host names if your account differs). No Java
install needed -- a portable JRE is already bundled at the repo root
(`../jre/`) and `config/sascfg_personal.py` points at it automatically.

In [ ]:
!test -f ~/.authinfo && echo "~/.authinfo found" || echo "missing -- create it in a terminal first, see the cell above"

In [ ]:
# Writes PROGRAM_SUMMARY, MACRO_PARAMS, DATA_DICTIONARY into your SASUSER
# library (SAS's auto-assigned, persistent-across-sessions library -- no
# LIBNAME statement or path to know). Pass --libname/--libpath only if you
# want a different, custom-path library instead. Needs the venv with
# saspy installed, hence the explicit interpreter path.
!/internal/venvs/main/bin/python3 push_to_oda.py --catalog ../results/catalog/config1-gemma-cpu

## Why zero-shot, not few-shot or fine-tuned

Tested in-context learning (showing the model 1-2 example SAS->JSON
pairs) here and it made output WORSE, not better: with even one exemplar
in context, `gemma3:1b` lost coherence and fabricated an entirely
fictional SAS program instead of documenting the real one. That's a
genuine capability ceiling of a 1B-parameter model on this task, not a
prompt bug -- if you want few-shot or fine-tuned quality, that's config2
(`../config2-qlora-gpu/`).

## Known limitations (report these honestly in the results)

- Small-model JSON-mode output frequently fails `schema.py`'s structural
  validation outright (missing keys, wrong types, a list entry that's a
  bare string instead of an object). `document_sas.py` does NOT crash on
  this -- it records `schema_valid: false` in the catalog/meta output and
  moves on, exactly so `results/score.py` can report schema validity as
  its own honest metric rather than one bad program aborting a batch run.
- Even when schema-valid, meanings/types are genuinely best-guess and
  were observed wrong on toy synthetic input (e.g. calling a sampling
  weight "Work Time"). The guardrail only catches INVENTED names, never
  wrong MEANINGS -- read every entry, don't just trust an absence of
  flags.
- `extract.py`'s regex scan is a heuristic on arbitrary real SAS syntax
  -- it can miss real identifiers written in forms it doesn't anticipate,
  which under-flags (reports clean when it isn't).